In [1]:
import glob
import h5py
import importlib
import IPython.display as ipd
import soxr
import numpy as np
import os
import pandas as pd
import pickle
import soundfile as sf
import src.audio_transforms as at
import src.custom_modules as cm
import src.spatial_attn_lightning as binaural_lightning
importlib.reload(binaural_lightning)
import sys
import torch
from tqdm import tqdm
import yaml

from pathlib import Path
from pytorch_lightning import Trainer
from scipy import signal
from scipy.io.wavfile import read, write

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [2]:
config = yaml.load(open('/om2/user/rphess/Auditory-Attention/config/binaural_attn/word_task_voice_cue.yml', 'r'), Loader=yaml.FullLoader)
mdl_ckpt = '/net/vast-storage/scratch/vast/mcdermott/rphess/Auditory-Attention/attn_cue_models/word_task_voice_cue/checkpoints/epoch=0-step=1250-v1.ckpt'

In [8]:
config['audio']['rep_kwargs']['rep_on_gpu'] = True
config['hparas']['batch_size'] = 90
config['num_workers'] = 10

In [4]:
corpora_config = config['corpus']

In [5]:
model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=mdl_ckpt, config=config)

num_classes={'num_words': 800}
Model performing word task
cochlea_filt {'sr': 50000, 'env_sr': 10000, 'n_channels': 40, 'low_lim': 40, 'use_pad': True, 'binaural': True, 'rep_on_gpu': True, 'center_crop': True, 'out_dur': 2, 'impulse_len': 0.25, 'env_extraction_type': 'Half-wave Rectification', 'downsampling_type': 'TorchTransformsResample', 'downsampling_kwargs': {'lowpass_filter_width': 64, 'rolloff': 0.9475937167399596, 'resampling_method': 'kaiser_window', 'beta': 14.769656459379492}} coch_p3 {'scale': 1, 'offset': 1e-07, 'clip_value': 5, 'power': 0.3}
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [ ]:
from re import L
import h5py
import torch
import glob
import sys
import os 
# sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch')
# import src.audio_transforms as audio_transforms
import pickle
import numpy as np


class BinauralAttentionDataset(torch.utils.data.ConcatDataset):
    # Makes a dataset using pre-paired speech and audioset background sounds
    
    def __init__(self, root, cue_type, task, batch_size=1, skip_negative_elev=False, mode='train', with_cue_free=False, **kwargs):
        """
        Builds the pytorch hdf5 combined dataset from the files found in the
        specified root directory. 
        """
        self.hdf5_glob = '*.hdf5_chunk000' if with_cue_free else 'noise*.hdf5_chunk000' 

        if mode == 'train':
            self.all_hdf5_files = glob.glob(root + '/train/' + self.hdf5_glob)
            # screen dead files 
            self.all_hdf5_files = [fname for fname in self.all_hdf5_files  if os.path.getsize(fname) > 0]
        elif mode == 'val':
            self.all_hdf5_files = glob.glob(root + '/validation/' + self.hdf5_glob) 
        elif mode == 'test':
            self.all_hdf5_files = glob.glob(root + '/test/' + self.hdf5_glob) 

        # read files to skip from a file
        with open(root + '/bad_files.txt', 'r') as f:
            files_to_skip = [line.strip().split('/')[-1] for line in f.readlines()]
        # filter bad files from the dataset
        self.all_hdf5_files = [fname for fname in self.all_hdf5_files if fname.split('/')[-1] not in files_to_skip]

        print(f"{len( self.all_hdf5_files)} files in {mode} concat dataset")
        self.all_hdf5_datasets = [H5Dataset(h5_file, cue_type, task, batch_size, skip_negative_elev) for h5_file in self.all_hdf5_files]

        super().__init__(self.all_hdf5_datasets)

    def class_map(self):
        """
        Loads the mapping between the word IDX and human readable word map. 
        """
        class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
        return class_map


class H5Dataset(torch.utils.data.Dataset):
    def __init__(self, path, cue_type, task, batch_size, skip_negative_elev=False):
        """
        Builds a pytorch hdf5 dataset
        Args:
            path (str): location of the hdf5 dataset
            task (str): string indicating label keys to return 
        """
        self.file_path = path
        self.dataset = None
        self.task = task
        self.batch_size = batch_size
        self.skip_negative_elev = skip_negative_elev
        # if self.skip_negative_elev:
        #     print("Skipping negative elevations")
        # else:
        #     print("Including negative elevations")

        if cue_type == 'voice_and_location':
            self.cue_key = 'voice_cue_target_loc'
        elif cue_type == 'voice':
            self.cue_key = "voice_cue_center_loc"
        elif cue_type == "location":
            self.cue_key = "loc_cue"

        if self.task == 'word':
            self.label_key = 'word_int'
        if self.task == 'location':
            self.label_key = ('label_loc_target_azim', 'label_loc_target_elev')
        if self.task == 'word_and_location':
            self.label_key = ('word_int', 'label_loc_target_azim', 'label_loc_target_elev')

        with h5py.File(self.file_path, 'r', swmr=True) as file:
            self.dataset_len = len(file['target'])

    def azim_elev_to_label(self, azim, elev):
        if self.skip_negative_elev:
            return np.array(((elev / 10) * 72) + (azim / 5) + 1, dtype=np.int64)
        else:
            return np.array((((elev + 30) / 10) * 72) + (azim / 5), dtype=np.int64)

    def __getitem__(self, index):
        """
        Gets components of the hdf5 file that are used for training
        Args: 
            index (int): index into the hdf5 file
        Returns:
            [signal, target] : the training audio (signal) containing the preprocessing
              which may combine the foreground and background speech, and the target idx
              specified by target_keys. 
        """
        start = index * self.batch_size
        end = start + self.batch_size
        if self.dataset is None:
            self.dataset = h5py.File(self.file_path, 'r', swmr=True)

        cue = self.dataset[self.cue_key][start:end].transpose((0, 2, 1))
        if np.isnan(cue).all():
            cue[:] = 0
        foreground = self.dataset['target'][start:end].transpose((0, 2, 1))
        background = self.dataset['bg_scene'][start:end].transpose((0, 2, 1))

        # if self.skip_negative_elev and self.dataset['label_loc_target_elev'][start:end] < 0:
        #     return None, None, None, None

        if self.task == 'word':
            label = self.dataset[self.label_key][start:end]
        elif self.task == 'location':
            azim = self.dataset[self.label_key[0]][start:end]
            elev = self.dataset[self.label_key[1]][start:end]
            label = []
            for azim, elev in zip(azim, elev):
                label.append(self.azim_elev_to_label(azim, elev))
        else:
            word = self.dataset[self.label_key[0]][start:end]
            azim = self.dataset[self.label_key[1]][start:end]
            elev = self.dataset[self.label_key[2]][start:end]
            loc = []
            for azim, elev in zip(azim, elev):
                loc.append(self.azim_elev_to_label(azim, elev))
            label = []
            for word, loc in zip(word, loc):
                label = (word, loc)
        return cue, foreground, background, label

    def __len__(self):
        return self.dataset_len

In [9]:
dataset = model.dataset(**corpora_config, batch_size=config['hparas']['batch_size'], mode='train')
print(f"len training set = {len(dataset)}")
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=1,
    num_workers=config['num_workers'], 
    collate_fn=model._collate_fn,
    pin_memory=True,
    shuffle=False,
)

1068 files in train concat dataset
len training set = 35244


In [13]:
from pytorch_lightning import Trainer
trainer = Trainer(
    precision=32,
    # default_root_dir='test_log_dump/',
    # val_check_interval=1000,
#     limit_train_batches=0.,
    limit_val_batches=0.0,
    num_nodes=1,
    gpus=1,
    # detect_anomaly=True,
    accelerator="gpu",
)
trainer.fit(model, dataloader)

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/loops/utilities.py:91: PossibleUserWarning: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
  rank_zero_warn(
GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                   | Params
------------------------------------------------------------
0 | audio_transforms | AudioCompose           | 0     
1 | model            | AttnSequentialAttacker | 72.4 M
2 | loss_fn          | CrossEntropyLoss       | 0     
3 | train_acc        | Accuracy               | 0     
4 | valid_acc        | Accuracy               | 0     
5 | test_acc         | Accuracy               | 0     
6 | test_confusion   | Accuracy               | 0     
--------------------------------------------

Training: 0it [00:00, ?it/s]

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/trainer/trainer.py:726: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")


In [10]:
for _ in tqdm(dataloader):
    pass

  1%|          | 363/35244 [01:16<2:02:15,  4.75it/s]
Exception in thread Thread-7:
Traceback (most recent call last):
  File "/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/threading.py", line 973, in _bootstrap_inner


KeyboardInterrupt: 

    self.run()
  File "/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/threading.py", line 910, in run
    self._target(*self._args, **self._kwargs)
  File "/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in _pin_memory_loop
Traceback (most recent call last):
  File "/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/multiprocessing/queues.py", line 251, in _feed
    send_bytes(obj)
  File "/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/multiprocessing/connection.py", line 205, in send_bytes
    self._send_bytes(m[offset:offset + size])
  File "/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/multiprocessing/connection.py", line 416, in _send_bytes
    self._send(header + buf)
  File "/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/multiprocessing/connection.py", line 373, in _send
    n = write(self._handle, buf)
Broken